# 🚀 Advanced RAG Pipeline
**Features:**
- 📄 YouTube Transcript Extraction
- 🧠 Semantic Chunking (embedding similarity-based breakpoints)
- 🗄️ ChromaDB Vector Store (free, local, persistent)
- 🔍 Hybrid Search: Dense (ChromaDB) + Sparse (BM25)
- ⚡ Reciprocal Rank Fusion (RRF) for optimal re-ranking
- 🤖 Groq LLaMA 3.1 for answer generation

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    rank-bm25 \
    python-dotenv \
    tiktoken \
    nltk

print("✅ All packages installed successfully!")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import math
import numpy as np
import nltk
from dotenv import load_dotenv
from getpass import getpass

# LangChain core
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# LLM
from langchain_groq import ChatGroq

# Embeddings (free, local)
from langchain_huggingface import HuggingFaceEmbeddings

# Vector DB
from langchain_chroma import Chroma

# Transcript
from youtube_transcript_api import YouTubeTranscriptApi

# BM25 sparse retrieval
from rank_bm25 import BM25Okapi

# Semantic similarity
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK tokenizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# ── Load environment variables ──────────────────────────────────────────────
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("🔑 Enter your Groq API key (get one free at https://console.groq.com): ")
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Configuration loaded!")
print(f"   Groq API Key: {'*' * (len(GROQ_API_KEY) - 4) + GROQ_API_KEY[-4:]}")

## Cell 3 — YouTube Transcript Extraction

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract YouTube video ID from various URL formats."""
    patterns = [
        r'(?:v=|youtu\.be/|embed/|v/|watch\?v=|shorts/)([A-Za-z0-9_-]{11})',
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract video ID from URL: {url}")


def fetch_transcript(youtube_url: str) -> str:
    """Fetch and combine YouTube transcript into a single text string."""
    video_id = extract_video_id(youtube_url)
    print(f"📺 Fetching transcript for video ID: {video_id}")

    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-US', 'en-GB'])
    full_text = " ".join(entry['text'] for entry in transcript_list)

    # Clean up common transcript artifacts
    full_text = re.sub(r'\[.*?\]', '', full_text)   # Remove [Music], [Applause] etc.
    full_text = re.sub(r'\s+', ' ', full_text).strip()

    print(f"✅ Transcript extracted: {len(full_text):,} characters, ~{len(full_text.split()):,} words")
    return full_text


# ── Enter your YouTube URL here ─────────────────────────────────────────────
YOUTUBE_URL = input("🔗 Enter YouTube video URL: ").strip()

raw_transcript = fetch_transcript(YOUTUBE_URL)
print("\n📝 Transcript preview (first 500 chars):")
print("-" * 60)
print(raw_transcript[:500])

## Cell 4 — Semantic Chunking

Instead of splitting by fixed character count, we:
1. Split transcript into sentences
2. Embed each sentence
3. Find breakpoints where adjacent sentence similarity **drops below a threshold**
4. Merge sentences between breakpoints into semantically coherent chunks

In [ ]:
# ── Initialize Embedding Model ───────────────────────────────────────────────
print("📦 Loading HuggingFace embedding model (all-MiniLM-L6-v2)...")
print("   (First run: ~90MB download. Subsequent runs use local cache.)")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("✅ Embedding model loaded!")


# ── Semantic Chunking Function ───────────────────────────────────────────────
def semantic_chunker(
    text: str,
    breakpoint_percentile: float = 85,
    min_chunk_size: int = 100,
    max_chunk_size: int = 2000,
) -> list[Document]:
    """
    Split text into semantically coherent chunks using embedding similarity.

    Args:
        text: Raw input text
        breakpoint_percentile: Percentile threshold to detect topic shifts.
            Higher = fewer, larger chunks. Lower = more, smaller chunks. (default: 85)
        min_chunk_size: Minimum characters per chunk (merges tiny chunks)
        max_chunk_size: Maximum characters per chunk (splits huge chunks)

    Returns:
        List of LangChain Document objects
    """
    # Step 1: Sentence tokenization
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    print(f"\n🔢 Total sentences: {len(sentences)}")

    # Step 2: Embed all sentences in batches
    print("⚙️  Computing sentence embeddings...")
    sentence_embeddings = embeddings_model.embed_documents(sentences)
    sentence_embeddings = np.array(sentence_embeddings)

    # Step 3: Compute cosine similarity between consecutive sentences
    similarities = []
    for i in range(len(sentence_embeddings) - 1):
        sim = cosine_similarity(
            sentence_embeddings[i].reshape(1, -1),
            sentence_embeddings[i + 1].reshape(1, -1)
        )[0][0]
        similarities.append(sim)

    # Step 4: Find breakpoints where similarity drops below threshold
    threshold = np.percentile(similarities, 100 - breakpoint_percentile)
    breakpoints = [
        i + 1  # breakpoint AFTER sentence i
        for i, sim in enumerate(similarities)
        if sim < threshold
    ]
    print(f"📍 Similarity threshold: {threshold:.4f}")
    print(f"📍 Detected {len(breakpoints)} breakpoints (topic shifts)")

    # Step 5: Group sentences into chunks at breakpoints
    chunk_boundaries = [0] + breakpoints + [len(sentences)]
    raw_chunks = []
    for start, end in zip(chunk_boundaries[:-1], chunk_boundaries[1:]):
        chunk_text = " ".join(sentences[start:end]).strip()
        if chunk_text:
            raw_chunks.append(chunk_text)

    # Step 6: Post-process — merge tiny chunks, split huge chunks
    final_chunks = []
    buffer = ""

    for chunk in raw_chunks:
        if len(buffer) + len(chunk) < min_chunk_size:
            buffer = (buffer + " " + chunk).strip()
        else:
            if buffer:
                final_chunks.append(buffer)
            buffer = chunk

    if buffer:
        final_chunks.append(buffer)

    # Split chunks that exceed max_chunk_size by sentences
    processed_chunks = []
    for chunk in final_chunks:
        if len(chunk) <= max_chunk_size:
            processed_chunks.append(chunk)
        else:
            # Split oversized chunk into halves
            chunk_sents = sent_tokenize(chunk)
            mid = len(chunk_sents) // 2
            processed_chunks.append(" ".join(chunk_sents[:mid]))
            processed_chunks.append(" ".join(chunk_sents[mid:]))

    # Step 7: Convert to LangChain Documents with metadata
    documents = [
        Document(
            page_content=chunk,
            metadata={
                "chunk_id": idx,
                "chunk_size": len(chunk),
                "source": YOUTUBE_URL,
            }
        )
        for idx, chunk in enumerate(processed_chunks)
        if chunk.strip()
    ]

    return documents


# ── Run Semantic Chunking ────────────────────────────────────────────────────
print("\n🧠 Running semantic chunking...")
semantic_chunks = semantic_chunker(
    raw_transcript,
    breakpoint_percentile=85,   # Adjust: higher = bigger chunks
    min_chunk_size=150,
    max_chunk_size=2000,
)

print(f"\n✅ Created {len(semantic_chunks)} semantic chunks")
print(f"   Avg chunk size: {sum(len(c.page_content) for c in semantic_chunks) // len(semantic_chunks)} chars")
print(f"   Min chunk size: {min(len(c.page_content) for c in semantic_chunks)} chars")
print(f"   Max chunk size: {max(len(c.page_content) for c in semantic_chunks)} chars")

print("\n📄 Sample Chunks:")
print("-" * 60)
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({chunk.metadata['chunk_size']} chars)")
    print(chunk.page_content[:200] + ("..." if len(chunk.page_content) > 200 else ""))

## Cell 5 — ChromaDB Vector Store (Dense Index)

In [ ]:
import chromadb
import shutil

CHROMA_PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "rag_youtube_collection"

# ── Option: Reset the DB if re-indexing a new video ─────────────────────────
RESET_DB = True  # Set to False to reuse existing index

if RESET_DB and os.path.exists(CHROMA_PERSIST_DIR):
    shutil.rmtree(CHROMA_PERSIST_DIR)
    print("🗑️  Cleared existing ChromaDB collection.")

# ── Build ChromaDB Vector Store ──────────────────────────────────────────────
print("\n🗄️  Building ChromaDB vector store...")
print(f"   Embedding {len(semantic_chunks)} chunks — this may take a moment...")

vector_store = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PERSIST_DIR,
)

print(f"\n✅ ChromaDB vector store created!")
print(f"   Collection: '{COLLECTION_NAME}'")
print(f"   Persisted at: {os.path.abspath(CHROMA_PERSIST_DIR)}")
print(f"   Documents indexed: {vector_store._collection.count()}")

# Dense retriever: top-10 candidates
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

## Cell 6 — BM25 Sparse Index (Keyword Search)

In [ ]:
import re
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> list[str]:
    """Tokenize text for BM25: lowercase, split on non-alphanumeric, filter short tokens."""
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens if len(t) > 2]  # Remove very short tokens


# ── Build BM25 Index ─────────────────────────────────────────────────────────
print("📑 Building BM25 sparse index...")

chunk_texts = [doc.page_content for doc in semantic_chunks]
tokenized_corpus = [simple_tokenize(text) for text in chunk_texts]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 index built over {len(chunk_texts)} chunks")


def bm25_retrieve(query: str, top_k: int = 10) -> list[tuple[Document, float]]:
    """Retrieve top-k chunks using BM25 scoring."""
    query_tokens = simple_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(semantic_chunks[i], float(scores[i])) for i in top_indices]


print("\n🔍 BM25 retriever ready!")

## Cell 7 — Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

**RRF Formula:**
$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

where `k=60` (smoothing constant) and `rank_r(d)` is the rank of document `d` in retriever result list `r`.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[Document]],
    k: int = 60,
    top_n: int = 5,
) -> list[tuple[Document, float]]:
    """
    Merge multiple ranked document lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: List of ranked Document lists (one per retriever)
        k: Smoothing constant (default 60, from Cormack et al. 2009)
        top_n: Number of top results to return

    Returns:
        List of (Document, rrf_score) tuples, sorted by score descending
    """
    rrf_scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            # Use chunk_id as unique key
            doc_key = str(doc.metadata.get("chunk_id", doc.page_content[:50]))
            rrf_scores[doc_key] = rrf_scores.get(doc_key, 0.0) + 1.0 / (k + rank)
            doc_map[doc_key] = doc

    # Sort by RRF score descending
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_map[key], score) for key, score in sorted_docs[:top_n]]


def hybrid_retrieve(query: str, top_k: int = 10, top_n: int = 5) -> list[Document]:
    """
    Hybrid retrieval: combine ChromaDB dense + BM25 sparse via RRF.

    Args:
        query: User question
        top_k: Candidates per retriever
        top_n: Final documents to return after RRF

    Returns:
        Top-N ranked Documents
    """
    # ── Dense retrieval (ChromaDB semantic search) ───────────────────────────
    dense_results = dense_retriever.invoke(query)

    # ── Sparse retrieval (BM25 keyword search) ───────────────────────────────
    bm25_results_with_scores = bm25_retrieve(query, top_k=top_k)
    sparse_results = [doc for doc, score in bm25_results_with_scores]

    # ── RRF Fusion ───────────────────────────────────────────────────────────
    fused = reciprocal_rank_fusion(
        ranked_lists=[dense_results, sparse_results],
        k=60,
        top_n=top_n,
    )

    return [doc for doc, score in fused], [score for doc, score in fused]


# ── Test hybrid retrieval ────────────────────────────────────────────────────
test_query = input("\n🧪 Enter a test query to verify retrieval: ").strip()

retrieved_docs, rrf_scores = hybrid_retrieve(test_query, top_k=10, top_n=5)

print(f"\n🏆 Top {len(retrieved_docs)} chunks after RRF fusion:")
print("=" * 60)
for i, (doc, score) in enumerate(zip(retrieved_docs, rrf_scores), 1):
    print(f"\n[Rank {i}] RRF Score: {score:.6f} | Chunk ID: {doc.metadata['chunk_id']}")
    print(f"Content: {doc.page_content[:200]}..." if len(doc.page_content) > 200 else f"Content: {doc.page_content}")

## Cell 8 — Groq LLM + RAG Chain

In [ ]:
# ── Initialize Groq LLM ──────────────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0.1,
    max_tokens=1024,
    groq_api_key=GROQ_API_KEY,
)

# ── RAG Prompt Template ──────────────────────────────────────────────────────
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert assistant that answers questions based on YouTube video transcripts.
Use ONLY the provided context to answer the question. If the answer is not in the context, 
say "I couldn't find that information in the video transcript."

Be concise, accurate, and cite relevant parts of the transcript in your answer.

Context from video transcript:
──────────────────────────────
{context}
──────────────────────────────

Question: {question}

Answer:"""
)


def format_context(docs: list[Document]) -> str:
    """Format retrieved documents into a single context string."""
    return "\n\n".join(
        f"[Chunk {doc.metadata['chunk_id']}]:\n{doc.page_content}"
        for doc in docs
    )


def answer_question(question: str, top_k: int = 10, top_n: int = 5) -> dict:
    """
    Full RAG pipeline: hybrid retrieve → RRF → Groq LLM.

    Returns dict with 'answer', 'sources', and 'rrf_scores'.
    """
    # Step 1: Hybrid retrieval with RRF
    retrieved_docs, scores = hybrid_retrieve(question, top_k=top_k, top_n=top_n)

    # Step 2: Format context
    context = format_context(retrieved_docs)

    # Step 3: Generate answer
    chain = RAG_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    return {
        "answer": answer,
        "sources": retrieved_docs,
        "rrf_scores": scores,
    }


print("✅ Groq LLM and RAG chain initialized!")
print(f"   Model: llama-3.1-70b-versatile")
print(f"   Temperature: 0.1 (factual/grounded)")

## Cell 9 — Interactive Q&A

In [ ]:
def display_answer(result: dict, show_sources: bool = True) -> None:
    """Pretty-print the RAG answer and retrieved sources."""
    print("\n" + "═" * 60)
    print("🤖 ANSWER")
    print("═" * 60)
    print(result["answer"])

    if show_sources:
        print("\n" + "─" * 60)
        print("📚 Retrieved Sources (after RRF re-ranking):")
        print("─" * 60)
        for i, (doc, score) in enumerate(zip(result["sources"], result["rrf_scores"]), 1):
            print(f"\n  [{i}] Chunk {doc.metadata['chunk_id']} | RRF: {score:.6f}")
            preview = doc.page_content[:150]
            print(f"      {preview}..." if len(doc.page_content) > 150 else f"      {preview}")
        print("─" * 60)


# ── Interactive Q&A Loop ─────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("💬 Advanced RAG Q&A — Interactive Mode")
print("   Type 'quit' or 'exit' to stop")
print("   Type 'no sources' to hide source chunks")
print("═" * 60)

show_sources = True

while True:
    question = input("\n❓ Your question: ").strip()

    if not question:
        continue

    if question.lower() in ("quit", "exit", "q"):
        print("\n👋 Goodbye!")
        break

    if question.lower() == "no sources":
        show_sources = False
        print("   ℹ️  Source display disabled.")
        continue

    if question.lower() == "show sources":
        show_sources = True
        print("   ℹ️  Source display enabled.")
        continue

    print("\n⏳ Retrieving and generating answer...")
    try:
        result = answer_question(question, top_k=10, top_n=5)
        display_answer(result, show_sources=show_sources)
    except Exception as e:
        print(f"\n❌ Error: {e}")